# Compare ViT with Swin, CoAtNet, DeiT

                                                                                                                                                    
- Swin Transformer: https://arxiv.org/abs/2103.14030                                                                                                 
- DeiT (Data-efficient Image Transformers): https://arxiv.org/abs/2012.12877                                                                         
- CoAtNet: https://arxiv.org/abs/2106.04803                                                                                                          
- Attention Is All You Need: https://arxiv.org/abs/1706.03762
- timm repo (has all these models implemented): https://github.com/huggingface/pytorch-image-models
- The Annotated Transformer (Harvard NLP walkthrough): https://nlp.seas.harvard.edu/annotated-transformer/

### Key Ideas Oversimplified:

#### Swin Transformer

* **Shifted windows**. Instead of global attention (every patch attends to every patch), Swin gets attentoin within local windows. (for example 7x7 patches). Then, it shifts the window grid by half a window between layers do information can flow across boundaries. 
* **Hierarchichal feature maps.** unlike ViT which keeps the same resolution thrughout, Swin progressively merges patches. (Think of this as pooling in CNNs). This gives us multi-scale features which will become very important in detection and segmentation later on. 
* Overall, Swim gives efficiency and versatiltiy. You can now get comparable accuracy with local attention of you shift the windows in a clever way.


#### DEiT

* showed you can get competitive results with less data, as long as you have the right training recipe. String augmentation, good regularization, and some careful hyperparameter tuning. 
* **Distillation token**. DEiT will add a second special token on top of the cls_token to learn from a CNN model. The CNN can teach the ViT what to attend to during training.

#### CoAtNet

* Brings in the best of both worlds. CNNS are good at local geatures and have nice inductive biases. Transformers are good at global relationships. If you stack CNNs and transformer blocks, you get good results. 
* **Relative Attention**. Take away the absolute positional embeddings like we had in the vanilla ViT, and instead use relative position biases in attentnions scores. This generalizes nicely to different input sizes. 


Building on top of our ViT, we export everything we wrote in the first module to vit.py. We can import that work and build on top of it

In [35]:
from vit import ViT, PatchEmbedding, MultiHeadAttention, TransformerEncoder ,ClassificationHead , FeedForward                                                               

## Baseline Vanilla ViT on CIFAR-100

In [3]:
from torchvision.datasets import CIFAR100
import torchvision.transforms as transforms
import torch
import torch.nn as nn
import torch.nn.functional as F # This gives us the softmax()
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((56, 56)),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])

train_dataset = CIFAR100(root='./data', train=True, download=True, transform=transform)
test_dataset = CIFAR100(root='./data', train=False, download=True, transform=transform)

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using: {device}")

Using: mps


In [5]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [6]:
model = ViT(
      img_size=56, patch_size=7, num_hiddens=256,
      num_heads=8, num_classes=100, depth=2, dropout=0.1
  ).to(device)

In [14]:
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(50):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (logits.argmax(dim=-1) == labels).sum().item()
        total += labels.size(0)

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Acc: {correct/total:.4f}")

Epoch 1 | Loss: 2.4357 | Acc: 0.3688
Epoch 2 | Loss: 2.3237 | Acc: 0.3913
Epoch 3 | Loss: 2.2071 | Acc: 0.4130
Epoch 4 | Loss: 2.0948 | Acc: 0.4366
Epoch 5 | Loss: 1.9839 | Acc: 0.4634
Epoch 6 | Loss: 1.8739 | Acc: 0.4888
Epoch 7 | Loss: 1.7806 | Acc: 0.5083
Epoch 8 | Loss: 1.6779 | Acc: 0.5296
Epoch 9 | Loss: 1.5713 | Acc: 0.5564
Epoch 10 | Loss: 1.4927 | Acc: 0.5736
Epoch 11 | Loss: 1.3945 | Acc: 0.5970
Epoch 12 | Loss: 1.3134 | Acc: 0.6175
Epoch 13 | Loss: 1.2441 | Acc: 0.6333
Epoch 14 | Loss: 1.1617 | Acc: 0.6567
Epoch 15 | Loss: 1.1082 | Acc: 0.6674
Epoch 16 | Loss: 1.0372 | Acc: 0.6853
Epoch 17 | Loss: 0.9735 | Acc: 0.7040
Epoch 18 | Loss: 0.9356 | Acc: 0.7135
Epoch 19 | Loss: 0.8795 | Acc: 0.7267
Epoch 20 | Loss: 0.8522 | Acc: 0.7346
Epoch 21 | Loss: 0.7989 | Acc: 0.7504
Epoch 22 | Loss: 0.7693 | Acc: 0.7581
Epoch 23 | Loss: 0.7278 | Acc: 0.7704
Epoch 24 | Loss: 0.7080 | Acc: 0.7768
Epoch 25 | Loss: 0.6750 | Acc: 0.7863
Epoch 26 | Loss: 0.6605 | Acc: 0.7895
Epoch 27 | Loss: 0.63

## Adding windowed attention like SWIN

* Step1: Windowed attention 
* lets run a 56x56 image through the steps to understand which changes and where we need to make

In [18]:
class AttentionHead(nn.Module):
    def __init__(self, dim, head_dim, dropout=0.1):
        super().__init__()
        self.scale = head_dim**-0.5
        self.q = nn.Linear(dim, head_dim, bias=False)
        self.k = nn.Linear(dim, head_dim, bias=False)
        self.v = nn.Linear(dim, head_dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        q, k, v = self.q(x), self.k(x), self.v(x)
        attn = torch.matmul(q, k.transpose(-1, -2)) * self.scale
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v)
        return out
    
class MultiHeadAttention(nn.Module):
    # For windowed attention, we need to reshape before we pass to attention.
    # We need to run attention independently for each window.
    def __init__(self, dim, num_heads, dropout=0.1):
        super().__init__()
        assert dim % num_heads == 0, "dim must be divisible by num_heads"
        self.head_dim = dim // num_heads
        self.num_heads = num_heads
        self.heads = nn.ModuleList(
            [AttentionHead(dim, self.head_dim, dropout) for _ in range(num_heads)]
        )
        self.proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        head_outputs = [head(x) for head in self.heads]
        out = torch.cat(head_outputs, dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

In [29]:
x = torch.randn(2, 3, 56, 56)

In [30]:
patch_emb = PatchEmbedding(img_size=56, patch_size=7, num_hiddens=256)
patches = patch_emb(x)
print("After conv:", patches.shape)  # [1, 256, 8, 8]
# hidden dim is the size of the vector that represent each patch. So since we choose 256, we expect each patch to be a vector of len 256
patches_flat = patches.flatten(2).transpose(1, 2)
print("Flattened:", patches_flat.shape)  # [2, 64, 256] — 8x8=64 patches. Each one of the 64 patches is now represented by a 256 vector. 

After conv: torch.Size([2, 256, 8, 8])
Flattened: torch.Size([2, 64, 256])


In [31]:
head = AttentionHead(dim=256, head_dim=32)
q = head.q(patches_flat)
k = head.k(patches_flat)
v = head.v(patches_flat)
print("Q:", q.shape)  # [2, 64, 32]
print("K:", k.shape)  # [2, 64, 32]. Why 32? Because in MHA, we split the input. So each head recevies 32 of the 256. 32*8 = 256.
attn_scores = torch.matmul(q, k.transpose(-1, -2))
print("Attention matrix:", attn_scores.shape)  # [2, 64, 64] — every patch attends to every patch

Q: torch.Size([2, 64, 32])
K: torch.Size([2, 64, 32])
Attention matrix: torch.Size([2, 64, 64])


* these are the inputs to attention. 64 patches, each represented by a a 32-dimensional vector. Its 32 because each head receives 256/num_heads, so 256/8 = 32. 

In [ ]:
# The 64 patches are an 8x8 grid. Window size 4 means 2x2 windows of 4x4 patches each.
B, num_patches, dim = patches_flat.shape
grid_size = 8   # sqrt(64)
window_size = 4
grid = patches_flat.reshape(B, grid_size, grid_size, dim)
print("Grid:", grid.shape)  # [2, 8, 8, 256]

Grid: torch.Size([2, 8, 8, 256])


In [33]:
# Split into windows
# [B, 8, 8, dim] -> [B, 2, 4, 2, 4, dim] -> [B, 2, 2, 4, 4, dim]
windows = grid.reshape(B, grid_size // window_size, window_size, grid_size // window_size, window_size, dim)
windows = windows.permute(0, 1, 3, 2, 4, 5)
print("Windows rearranged:", windows.shape)  # [2, 2, 2, 4, 4, 256]

Windows rearranged: torch.Size([2, 2, 2, 4, 4, 256])


* The above reshaping. We have an 8x8 grid. But we specify window size of 4. So the 8x8 grid will now be 4 regions, each a 4x4 grid of original patches. So instead of every patch attentding to all others in the 8x8 grid, they only attend to the region, so 4x4. 


```
  ┌───────────┬───────────┐                                                                                                                            
  │           │           │
  │  window   │  window   │                                                                                                                            
  │  (0,0)    │  (0,1)    │
  │  4×4      │  4×4      │
  │           │           │
  ├───────────┼───────────┤
  │           │           │
  │  window   │  window   │
  │  (1,0)    │  (1,1)    │
  │  4×4      │  4×4      │
  │           │           │
  └───────────┴───────────┘
```

- B=2 — batch
- 2, 2 — which window (row, col) — 4 windows total
- 4, 4 — which patch within that window — 16 patches per window
- 256 — hidden dim

In [34]:
# Merge batch and window dims so we can run attention as-is
num_windows = (grid_size // window_size) ** 2  # 4
windows = windows.reshape(B * num_windows, window_size * window_size, dim)
print("Ready for attention:", windows.shape)  # [8, 16, 256] — 8 = 2 batches * 4 windows, 16 patches each

# Step 4: Run attention on windowed input
attn_out = head(windows)
print("Attention output:", attn_out.shape)  # [8, 16, 32]

# Step 5: Reverse the reshape to get back to [B, 64, dim]
# [8, 16, 32] -> [2, 2, 2, 4, 4, 32] -> [2, 8, 8, 32] -> [2, 64, 32]
attn_out = attn_out.reshape(B, grid_size // window_size, grid_size // window_size, window_size, window_size, -1)
attn_out = attn_out.permute(0, 1, 3, 2, 4, 5)
attn_out = attn_out.reshape(B, num_patches, -1)
print("Back to original shape:", attn_out.shape)  # [2, 64, 32]

Ready for attention: torch.Size([8, 16, 256])
Attention output: torch.Size([8, 16, 32])
Back to original shape: torch.Size([2, 64, 32])


### So all we have to do is reshape in MHA

In [54]:
class MultiHeadAttentionShift(nn.Module):
    # For windowed attention, we need to reshape before we pass to attention.
    # We need to run attention independently for each window.
    def __init__(self, dim, num_heads, grid_size, 
                  window_size, dropout=0.1):
        super().__init__()
        assert dim % num_heads == 0, "dim must be divisible by num_heads"
        self.head_dim = dim // num_heads
        self.grid_size = grid_size
        self.window_size = window_size
        self.num_heads = num_heads
        self.heads = nn.ModuleList(
            [AttentionHead(dim, self.head_dim, dropout) for _ in range(num_heads)]
        )
        self.proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, num_patches_with_cls, dim = x.shape
        cls_token = x[:, :1, :]
        x = x[:, 1:, :]
        num_patches = x.shape[1]
        num_windows = (self.grid_size // self.window_size) ** 2

        # Window the patches
        grid = x.reshape(B, self.grid_size, self.grid_size, dim)
        windows = grid.reshape(B, self.grid_size // self.window_size, self.window_size,
                                    self.grid_size // self.window_size, self.window_size, dim)
        windows = windows.permute(0, 1, 3, 2, 4, 5)
        windows = windows.reshape(B * num_windows, self.window_size * self.window_size, dim)

        # Append CLS to every window so it can attend to all patches
        cls_expanded = cls_token.repeat(num_windows, 1, 1)  # [B*num_windows, 1, dim]
        windows = torch.cat((cls_expanded, windows), dim=1)  # [B*num_windows, 17, dim]

        # Run attention
        head_outputs = [head(windows) for head in self.heads]
        out = torch.cat(head_outputs, dim=-1)

        # Separate CLS and patches
        cls_out = out[:, :1, :]    # [B*num_windows, 1, dim]
        out = out[:, 1:, :]        # [B*num_windows, 16, dim]

        # Average CLS across all windows (each window produced a CLS update)
        cls_out = cls_out.reshape(B, num_windows, 1, dim).mean(dim=1)  # [B, 1, dim]

        # Reverse windowing for patches
        nw = self.grid_size // self.window_size
        out = out.reshape(B, nw, nw, self.window_size, self.window_size, dim)
        out = out.permute(0, 1, 3, 2, 4, 5)
        out = out.reshape(B, num_patches, dim)

        # Recombine
        out = torch.cat((cls_out, out), dim=1)
        out = self.proj(out)
        out = self.dropout(out)
        return out
    
class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_dim,
                 grid_size, window_size = None, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MultiHeadAttentionShift(dim, num_heads, grid_size, window_size, dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = FeedForward(dim, mlp_dim, dropout)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x
    
class TransformerEncoder(nn.Module):
    def __init__(self, dim, depth, num_heads, mlp_dim, grid_size, window_size = None, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList(
            [TransformerBlock(dim, num_heads, mlp_dim,grid_size, 
                              window_size, dropout) for _ in range(depth)]
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        for block in self.layers:
            x = block(x)
        return self.norm(x)

In [55]:
class ViTWindowed(nn.Module):
      def __init__(self, img_size, patch_size, num_hiddens, num_heads, 
                   num_classes, window_size, depth=6, dropout=0.1):
          super().__init__()
          self.num_patches = (img_size // patch_size) ** 2
          self.grid_size = img_size // patch_size

          # --- Patch Embedding ---
          self.patch_embedding = PatchEmbedding(img_size, patch_size, num_hiddens)

          # --- CLS token ---
          self.cls_token = nn.Parameter(torch.randn(1, 1, num_hiddens))

          # --- Positional Embedding (+1 for CLS) ---
          self.pos_embedding = nn.Parameter(torch.randn(1, self.num_patches + 1, num_hiddens))

          # --- Transformer Encoder ---
          self.encoder = TransformerEncoder(num_hiddens, depth, num_heads,  
                                            mlp_dim=num_hiddens*4,
                                            grid_size=self.grid_size,
                                            window_size=window_size)

          # --- Classification Head ---
          self.classification_head = ClassificationHead(num_hiddens, num_classes)

      def forward(self, x):
          batch = x.shape[0]

          # 1. Patch embed: [B, 3, H, W] -> [B, num_patches, num_hiddens]
          patches = self.patch_embedding(x)
          patches = patches.flatten(2).transpose(1, 2)

          # 2. Prepend CLS token: [B, num_patches, D] -> [B, num_patches+1, D]
          cls_tokens = self.cls_token.expand(batch, -1, -1)
          x = torch.cat((cls_tokens, patches), dim=1)

          # 3. Add positional embeddings
          x = x + self.pos_embedding

          # 4. Transformer encoder
          x = self.encoder(x)

          # 5. Classification: extract CLS token -> head
          x = self.classification_head(x)

          return x  # placeholder — returns full sequence for now

In [58]:
model = ViTWindowed(
      img_size=56, patch_size=7, num_hiddens=256,
      num_heads=8, num_classes=100,window_size =4, depth=2, dropout=0.1
  ).to(device)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(50):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (logits.argmax(dim=-1) == labels).sum().item()
        total += labels.size(0)

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Acc: {correct/total:.4f}")

Epoch 1 | Loss: 3.7393 | Acc: 0.1237
Epoch 2 | Loss: 3.1742 | Acc: 0.2189
Epoch 3 | Loss: 2.9502 | Acc: 0.2607
Epoch 4 | Loss: 2.7833 | Acc: 0.2934
Epoch 5 | Loss: 2.6495 | Acc: 0.3208
Epoch 6 | Loss: 2.5246 | Acc: 0.3440
Epoch 7 | Loss: 2.4129 | Acc: 0.3681
Epoch 8 | Loss: 2.2987 | Acc: 0.3920
Epoch 9 | Loss: 2.1914 | Acc: 0.4161
Epoch 10 | Loss: 2.0869 | Acc: 0.4407
Epoch 11 | Loss: 1.9806 | Acc: 0.4627
Epoch 12 | Loss: 1.8883 | Acc: 0.4816
Epoch 13 | Loss: 1.7950 | Acc: 0.5023
Epoch 14 | Loss: 1.7079 | Acc: 0.5217
Epoch 15 | Loss: 1.6260 | Acc: 0.5414
Epoch 16 | Loss: 1.5424 | Acc: 0.5596
Epoch 17 | Loss: 1.4716 | Acc: 0.5779
Epoch 18 | Loss: 1.3945 | Acc: 0.5956
Epoch 19 | Loss: 1.3194 | Acc: 0.6141
Epoch 20 | Loss: 1.2579 | Acc: 0.6280
Epoch 21 | Loss: 1.1985 | Acc: 0.6420
Epoch 22 | Loss: 1.1353 | Acc: 0.6605
Epoch 23 | Loss: 1.0895 | Acc: 0.6670
Epoch 24 | Loss: 1.0241 | Acc: 0.6884
Epoch 25 | Loss: 0.9777 | Acc: 0.6979
Epoch 26 | Loss: 0.9413 | Acc: 0.7085
Epoch 27 | Loss: 0.90

* So we have the computational advantage of windowed attention, but clearly by removing the global context we have hurt accuracy.

* According to the SWIN paper, thats why we need the shifted windows, so patches can actually have some overlap.

## Lets add shifted windows now

## Check how the pros do it

[Timm Swift Implementation](https://github.com/huggingface/pytorch-image-models/blob/a94c10fce182362e26e128e1b51863dff2a1d558/timm/models/swin_transformer.py#L42-L58)

and their [SwinTransformer Block](https://github.com/huggingface/pytorch-image-models/blob/a94c10fce182362e26e128e1b51863dff2a1d558/timm/models/swin_transformer.py#L255)

* timm patritions windows like this
  
* this actually supports non-square windows, which is a nice idea. 

```python
def window_partition(
        x: torch.Tensor,
        window_size: Tuple[int, int],
) -> torch.Tensor:
    """Partition into non-overlapping windows.

    Args:
        x: Input tokens with shape [B, H, W, C].
        window_size: Window size.

    Returns:
        Windows after partition with shape [B * num_windows, window_size, window_size, C].
    """
    B, H, W, C = x.shape
    x = x.view(B, H // window_size[0], window_size[0], W // window_size[1], window_size[1], C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size[0], window_size[1], C)
    return windows
```


And this is how do they do window reverse:

```python
def window_reverse(windows: torch.Tensor, window_size: Tuple[int, int], H: int, W: int) -> torch.Tensor:
    """Reverse window partition.

    Args:
        windows: Windows with shape (num_windows*B, window_size, window_size, C).
        window_size: Window size.
        H: Height of image.
        W: Width of image.

    Returns:
        Tensor with shape (B, H, W, C).
    """
    C = windows.shape[-1]
    x = windows.view(-1, H // window_size[0], W // window_size[1], window_size[0], window_size[1], C)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, H, W, C)
    return x
```
